In [21]:
import numpy as np
import pandas as pd
import random

In [22]:
def calculate_similarity(arr, prev_pct, lamda):
    return (np.sum(arr)**2)/(np.sum(np.multiply(prev_pct, 1-prev_pct)) + lamda )
def calculate_cover(prev_pct):
    return np.sum(np.multiply(prev_pct, 1-prev_pct))

def sigmoid(log_odds):
    res = 1/ (1+ np.exp(-log_odds))
    return res

def best_gain(X, prev_pct, y, lamda, gamma, cover):
    best_score, best_col, best_thresh = -np.inf, None, None
    root_similarity = calculate_similarity(y, prev_pct, lamda)

    for col in range(X.shape[1]):
        s = np.sort(np.unique(X[:, col]))
        thresholds = [(s[i] + s[i+1])/2 for i in range(len(s) - 1)]

        for thresh in thresholds:
            score = split_score(X, prev_pct, y, col,root_similarity, thresh, lamda, cover)
            if score > gamma and score > best_score:
                best_score, best_col, best_thresh = score, col, thresh
                
    return (best_col, best_score, best_thresh)
        
def split_score(X, prev_pct, y, feature_column, root_similarity, threshold, lamda, cover):
    true_array, true_pct = y[X[:, feature_column] >= threshold], prev_pct[X[:, feature_column] >= threshold]
    false_array, false_pct = y[X[:, feature_column] < threshold], prev_pct[X[:, feature_column] < threshold]
    if calculate_cover(true_pct) < cover or calculate_cover(false_pct) < cover:
        return -np.inf

    true_res = calculate_similarity(true_array, true_pct, lamda)
    false_res = calculate_similarity(false_array, false_pct, lamda)

    return true_res + false_res - root_similarity

In [23]:
from __future__ import annotations
from dataclasses import dataclass

@dataclass
class LeafNode:
    res: float

@dataclass
class DecisionNode:
    col: int
    threshold: float
    leftNode: DecisionNode | LeafNode | None = None
    rightNode: DecisionNode | LeafNode | None = None

In [24]:
def _build(X, prev_pct, y, depth = 2, score_cutoff = 0, lamda = 1, gamma = 0, cover = 1):   
    if not len(X) or not len(y):
        raise IndexError("Length was 0?")
    
    best_col, best_score, thresh = best_gain(X, prev_pct, y, lamda, gamma, cover)
    curr = DecisionNode(best_col, thresh)

    if depth == 0 or best_col == None:
        mean = np.sum(y)/(calculate_cover(prev_pct) + lamda)
        return LeafNode(mean)
    else:
        left_x, left_y, left_pct = X[X[:,best_col] >= thresh], y[X[:,best_col] >= thresh], prev_pct[X[:,best_col] >= thresh]
        right_x, right_y, right_pct = X[X[:,best_col] < thresh], y[X[:,best_col] < thresh],prev_pct[X[:,best_col] < thresh]
        curr.leftNode = _build(left_x, left_pct, left_y, depth-1, score_cutoff, lamda =lamda, gamma = gamma, cover = cover)
        curr.rightNode = _build(right_x, right_pct, right_y, depth-1, score_cutoff, lamda = lamda, gamma = gamma, cover = cover)
    return curr

def predict(X, tree):
    ret = []
    for sample in X:
        curr = tree
        while curr and type(curr) is DecisionNode:
            c = curr.col
            if sample[c] >= curr.threshold:
                curr = curr.leftNode
            else:
                curr = curr.rightNode
        ret.append(curr.res if curr else None)
    return ret

def create_xgboost(X, y, depth, lr=0.1, num_trees = 3, lamda = 0, gamma = 0, cover = 1):
    ret = [np.log(len(y[y==1])/len(y[y == 0]))]
    for _ in range(num_trees):
        res = ret[0] + lr * xgboost_predict(X,ret)
        pct_res = np.array([sigmoid(r) for r in res])
        new_y = y - pct_res
        ret.append(_build(X, pct_res, new_y, depth = depth, lamda=lamda, gamma=gamma, cover = cover))
    return ret

def xgboost_predict(X, f, lr = 0.1):
    curr = np.zeros(X.shape[0])
    for tree in f[1:]:
        curr += predict(X, tree)
    return curr

In [25]:
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestClassifier
from collections import deque

def _display(curr: DecisionNode | LeafNode | None):
    if type(curr) is DecisionNode:
        print(f"col: {curr.col}, threshold: {curr.threshold:.2f}", end = " | ")
    elif type(curr) is LeafNode:
        print(f"res: {curr.res}", end = " | ")
    else:
        print(f"res : {None}")
        
def _display_graph(tree):
    q, newq = deque([tree]), deque([])
    while q or newq:
        while q:
            curr = q.pop()
            _display(curr)
            if not curr or type(curr) is not DecisionNode: continue
            for item in [curr.leftNode, curr.rightNode]:
                newq.appendleft(item)
        q, newq = newq, deque()
        print("", end ="\n")

In [26]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

X,y = make_classification(n_samples = 200, n_features = 4, random_state = 0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

xgb = XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3,
    reg_lambda=1.0, gamma=0.0, min_child_weight=1.0,
    tree_method="exact",
    base_score=float(y_train.mean()),   # a probability; xgboost converts it to log-odds
)
xgb.fit(X_train, y_train)
local_gb = create_xgboost(X_train, y_train, num_trees=100, depth = 3, lamda = 1, gamma= 0)

sk_preds = xgb.predict(X_test)
sk_preds_proba = xgb.predict_proba(X_test)[:, 1]
my_preds = xgboost_predict(X_test, local_gb)
my_preds = local_gb[0] + 0.1 * my_preds 
my_preds = np.array([sigmoid(s) for s in my_preds])

print(mean_squared_error(y_test, sk_preds), r2_score(y_test, sk_preds))
print(mean_squared_error(y_test, my_preds), r2_score(y_test, my_preds))
print()
print(np.max(np.abs(sk_preds_proba - my_preds)))
print(np.sum(sk_preds ^ (my_preds > 0.5)))
print()
print(classification_report(y_test, sk_preds))
print(classification_report(y_test, my_preds > 0.5))


0.06 0.7596153846153846
0.059016895342440485 0.7635541051985557

5.622800158811003e-08
0

              precision    recall  f1-score   support

           0       0.96      0.92      0.94        26
           1       0.92      0.96      0.94        24

    accuracy                           0.94        50
   macro avg       0.94      0.94      0.94        50
weighted avg       0.94      0.94      0.94        50

              precision    recall  f1-score   support

           0       0.96      0.92      0.94        26
           1       0.92      0.96      0.94        24

    accuracy                           0.94        50
   macro avg       0.94      0.94      0.94        50
weighted avg       0.94      0.94      0.94        50

